# GI findings → Excel (report-team handoff)

1. Pick a **client** (GI) and associated **report JSON** files
2. Run PolicyGuard
3. Write Excel under `data/pipeline/findings/` with columns:

`Order # | Empty | 1 | What was checked | Checkpoint | Original Content | Non-confirmities | Suggested actions | Finding Verdict | Remark`

**Finding Verdict** and **Remark** are left blank for the report team.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from clients import ensure_pipeline_dirs, list_clients, load_client
from findings_export import (
    checkpoint_meta_by_id,
    order_id_for_report,
    requirements_by_id,
    write_findings_workbook,
)
from gi_review import format_cost_summary, load_checkpoints
from review import run_production_review

os.environ.setdefault("GI_PIPELINE_LOG", "0")

# --- config ---
CLIENT = "ribkoff"  # one of list_clients()
# July14 batch
REPORTS = sorted(
    str(p) for p in (PROJECT_ROOT / "data/clients/July14_ribkoff").glob("Q*.json")
)
USE_FLAWED = False  # True → run labeled flawed reports instead
ENABLE_VISION = False
OUT_DIR = PROJECT_ROOT / "data/pipeline/findings"

ensure_pipeline_dirs(PROJECT_ROOT)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("clients:", list_clients(PROJECT_ROOT))
print("CLIENT:", CLIENT)
print("REPORTS:", len(REPORTS))
print("OUT_DIR:", OUT_DIR)


In [ ]:
client = load_client(CLIENT, PROJECT_ROOT)
if not client.checkpoints_path.exists():
    raise FileNotFoundError(f"Missing checkpoints: {client.checkpoints_path}")

checkpoints = load_checkpoints(client.checkpoints_path)
requirements = requirements_by_id(checkpoints)
meta_by_id = checkpoint_meta_by_id(checkpoints)

if USE_FLAWED:
    report_paths = [p for p, _key in client.flawed_reports()]
else:
    report_paths = client.corrected_reports()
    if REPORTS:
        wanted = {Path(r).stem for r in REPORTS}
        report_paths = [p for p in report_paths if p.stem in wanted]
        # also allow full paths in REPORTS
        for r in REPORTS:
            rp = Path(r)
            if not rp.is_absolute():
                rp = PROJECT_ROOT / rp
            if rp.exists() and rp.suffix == ".json" and rp not in report_paths:
                report_paths.append(rp)

if not report_paths:
    raise FileNotFoundError("No reports selected — check CLIENT / REPORTS / USE_FLAWED")

print(f"{len(checkpoints)} checkpoints | {len(report_paths)} reports")
for p in report_paths:
    print(" -", p.relative_to(PROJECT_ROOT) if p.is_relative_to(PROJECT_ROOT) else p)


In [ ]:
orders = []  # (order_id, flags)
summaries = []
reports_by_order = {}  # for Original Content enrichment from live JSON

for report_path in report_paths:
    t0 = time.perf_counter()
    report = json.loads(report_path.read_text(encoding="utf-8"))
    run = run_production_review(
        report_path,
        client.checkpoints_path,
        specs_path=client.hand_specs,
        project_root=PROJECT_ROOT,
        enable_vision=ENABLE_VISION,
    )
    summary = format_cost_summary(run)
    elapsed = round(time.perf_counter() - t0, 1)
    order_id = order_id_for_report(report_path, report)
    flags = list(summary.get("flags") or [])
    orders.append((order_id, flags))
    reports_by_order[order_id] = report
    summaries.append(
        {
            "order_id": order_id,
            "report": report_path.name,
            "blocking": summary.get("blocking_flags_count"),
            "total_flags": summary.get("total_flags_count"),
            "unable": summary.get("unable_to_check_count"),
            "elapsed_s": elapsed,
            "cost_usd": (summary.get("cost_usd") or {}).get("total", {}).get("total_cost_usd"),
        }
    )
    # also stash machine JSON next to excel for debugging
    dump = OUT_DIR / f"{order_id}_review.json"
    dump.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    print(
        f"{order_id}: flags={len(flags)} blocking={summary.get('blocking_flags_count')} "
        f"unable={summary.get('unable_to_check_count')} {elapsed}s → {dump.name}"
    )

print("done", len(orders), "orders")


In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_xlsx = OUT_DIR / f"{CLIENT}_findings_{stamp}.xlsx"
write_findings_workbook(
    orders,
    requirements,
    out_xlsx,
    summary_rows=summaries,
    meta_by_id=meta_by_id,
    reports_by_order=reports_by_order,
)
print("Wrote", out_xlsx)
print("Sheets: Findings | Summary | Verdict legend")
total = sum(len(f) for _, f in orders)
print(f"{total} findings across {len(orders)} orders")
